# 포켓몬 배틀 게임 만들기 — 1단계

## 타입 상성 시스템

오늘 만들 것
- 불꽃🔥·물💧·풀🌿 같은 타입을 안전하게 정의하기 (Enum)
- "불꽃🔥은 풀🌿에 강하다" 같은 상성을 표로 정리하기 (딕셔너리)
- 포켓몬끼리 공격하고 데미지를 계산하기 (클래스)
- 가끔 데미지가 커지는 급소 넣기 (random)
- 마지막에 두 포켓몬이 끝까지 싸우는 배틀 만들기

새로 배우는 도구: `random`, `@dataclass`


## 1. 도입 — if-elif로 상성을 만들면?

"불꽃이 풀을 공격하면 데미지 2배" 같은 규칙을 우리가 아는 if-elif로만 적어 보자.
타입이 늘어날수록 이 방식이 어떻게 되는지 직접 확인해 본다.


In [ ]:
# 공격 타입과 방어 타입을 받아 데미지 배수를 정해주는 함수 (if-elif 버전)
def get_multiplier_ifelif(attack_type, defend_type):
    if attack_type == "불꽃" and defend_type == "풀":
        return 2.0                      # 불꽃 -> 풀: 효과 굉장
    elif attack_type == "불꽃" and defend_type == "물":
        return 0.5                      # 불꽃 -> 물: 별로
    elif attack_type == "물" and defend_type == "불꽃":
        return 2.0
    elif attack_type == "풀" and defend_type == "물":
        return 2.0
    else:
        return 1.0                      # 나머지는 보통(1배)

print(get_multiplier_ifelif("불꽃", "풀"))   # 2.0
print(get_multiplier_ifelif("불꽃", "물"))   # 0.5
print(get_multiplier_ifelif("물", "불꽃"))   # 2.0


### 무엇이 문제일까

포켓몬 타입은 보통 18개다. 모든 조합을 if-elif로 적으면 18 × 18 = 324줄이 된다.

- 손으로 적다가 오타가 나기 쉽다.
- 규칙 하나를 바꾸려면 그 줄을 일일이 찾아야 한다.
- 코드가 길어서 읽기 어렵다.

해결 방법: 타입은 **Enum**으로 안전하게 정의하고, 상성은 **딕셔너리**로 표처럼 정리한다.


## 2. 개념 ① Enum — 정해진 값만 쓰게 하기

타입을 `"불꽃"`, `"물"` 같은 문자열로 적으면 두 가지 문제가 생긴다.

- **오타**: `"불곷"`이라고 잘못 써도 그냥 실행돼서, 한참 뒤에야 버그를 발견한다.
- **모호함**: 같은 타입을 누구는 `"불꽃"`, 누구는 `"화염"`, `"파이어"`로 적으면 표가 어긋난다.

**Enum**(enumeration, 열거)은 *쓸 수 있는 값을 미리 정해 두는 묶음*이다.
한 번 정해 두면 그 안의 값만 쓸 수 있고, 없는 값을 쓰면 그 자리에서 오류가 난다.
그래서 오타를 일찍 잡을 수 있다.

이 수업에서 쓸 타입: 🔥 불꽃, 💧 물, 🌿 풀, ⚡ 전기, 노말, 땅
(아이콘은 이해를 돕기 위한 것이고, 코드에서는 이름과 값으로 다룬다.)

꺼내 쓰는 법은 점(`.`)을 찍는다.
- `Type.FIRE.name` → `"FIRE"` (영어 이름)
- `Type.FIRE.value` → `"불꽃"` (우리가 정해 둔 값)


In [ ]:
from enum import Enum  # Enum 사용

# 포켓몬 타입 정의. 이제 이 6개 말고는 쓸 수 없다.
class Type(Enum):
    FIRE = "불꽃"
    WATER = "물"
    GRASS = "풀"
    ELECTRIC = "전기"
    NORMAL = "노말"
    GROUND = "땅"

print(Type.FIRE)            # Type.FIRE
print(Type.FIRE.name)       # FIRE
print(Type.FIRE.value)      # 불꽃


In [ ]:
# Enum이 오타를 어떻게 막는지 확인 (배운 예외처리 활용)
try:
    Type.FIREE          # 일부러 오타 (FIRE 를 FIREE 로)
except AttributeError:
    print("그런 타입은 없음 -> Enum이 오타를 바로 잡아준다")
    print("문자열이었다면 오류 없이 넘어가 버그가 됐을 것")


## 3. 개념 ② 상성표 — 중첩 딕셔너리와 `.get()`

상성을 표로 생각하면 정리하기 쉽다.

| 공격 \ 방어 | 🔥 불꽃 | 💧 물 | 🌿 풀 |
|:---:|:---:|:---:|:---:|
| **🔥 불꽃** | 0.5 | 0.5 | 2.0 |
| **💧 물** | 2.0 | 0.5 | 0.5 |
| **🌿 풀** | 0.5 | 2.0 | 0.5 |

표를 세로(공격)로 읽는다. 예: 불꽃🔥으로 풀🌿을 치면 2.0배, 물💧을 치면 0.5배다.

이 표를 코드로 옮기면 딕셔너리 안에 딕셔너리가 들어간 모양이 된다.
`{ 공격타입: { 방어타입: 배수 } }` — 이것을 **중첩 딕셔너리**라고 한다.

표에 적지 않은 조합(예: 노말 → 노말)은 보통, 즉 1.0배로 두고 싶다.
이때 **`.get(키, 기본값)`** 을 쓰면 키가 없을 때 기본값을 돌려준다.
'있는지 없는지'를 if로 확인하지 않아도 된다.

그래서 `get_multiplier`는 두 단계로 동작한다.
1. 공격 타입의 작은 표를 꺼낸다. 없으면 빈 표 `{}`.
2. 그 표에서 방어 타입을 찾는다. 없으면 `1.0`.


예상 출력
```
불꽃 -> 풀 : 2.0
불꽃 -> 물 : 0.5
물 -> 불꽃 : 2.0
전기 -> 전기 : 1.0   (표에 없어 기본값)
```


In [ ]:
# 상성표: { 공격타입: { 방어타입: 배수 } }
TYPE_CHART = {
    Type.FIRE: {Type.GRASS: 2.0, Type.WATER: 0.5, Type.FIRE: 0.5},
    Type.WATER: {Type.FIRE: 2.0, Type.GRASS: 0.5, Type.WATER: 0.5},
    Type.GRASS: {Type.WATER: 2.0, Type.FIRE: 0.5, Type.GRASS: 0.5},
    Type.ELECTRIC: {Type.WATER: 2.0, Type.GROUND: 0.0},   # 전기는 땅에 안 통함(0배)
}

def get_multiplier(attack_type, defend_type):
    sub_table = TYPE_CHART.get(attack_type, {})   # 1) 공격 타입의 표, 없으면 {}
    return sub_table.get(defend_type, 1.0)        # 2) 방어 타입의 배수, 없으면 1.0

print("불꽃 -> 풀 :", get_multiplier(Type.FIRE, Type.GRASS))       # 2.0
print("불꽃 -> 물 :", get_multiplier(Type.FIRE, Type.WATER))       # 0.5
print("물 -> 불꽃 :", get_multiplier(Type.WATER, Type.FIRE))       # 2.0
print("전기 -> 전기 :", get_multiplier(Type.ELECTRIC, Type.ELECTRIC))  # 1.0


### 직접 해보기 ① — 가장 효과적인 공격 타입 찾기

상성표와 `get_multiplier`, 그리고 반복문을 **조합**하는 문제다.

방어 타입 하나를 받아서, 후보(불꽃🔥·물💧·풀🌿·전기⚡) 중 데미지 배수가 가장 큰 공격 타입을
찾아 돌려주는 `best_attack_against(defend_type)` 를 완성해 보자.

생각할 점: 후보를 하나씩 보면서 배수를 구하고, 지금까지 본 것 중 가장 큰 값을 기억해 둔다.

예) 방어가 물💧이면 → 풀🌿 또는 전기⚡(배수 2.0)가 가장 효과적이다.


In [ ]:
def best_attack_against(defend_type):
    candidates = [Type.FIRE, Type.WATER, Type.GRASS, Type.ELECTRIC]
    best_type = None
    best_mult = -1                       # 아직 아무것도 못 봤으니 가장 작은 값에서 시작
    for atk in candidates:
        m = get_multiplier(atk, defend_type)
        if m > best_mult:                # 지금까지 중 가장 크면 갱신
            best_mult = m
            best_type = atk
    return best_type, best_mult

t, m = best_attack_against(Type.WATER)
print(f"물에게 가장 효과적: {t.value} ({m}배)")


## 4. 실습 — Pokemon 클래스

포켓몬 한 마리는 이름, 타입, HP, 공격력을 가진다.
`attack(target)` 은 상성표에서 배수를 가져와 데미지를 계산하고 상대의 HP를 깎는다.

- 데미지 = 공격력 × 상성 배수
- 배수 > 1 → "효과가 굉장했다"
- 배수 < 1 → "효과는 별로였다"
- 배수 = 1 → "그럭저럭 통했다"


예상 출력 (전기 → 물은 2배, 공격력 20 × 2 = 40)
```
피카츄의 공격! 꼬부기에게 40 데미지
효과가 굉장했다
꼬부기의 남은 HP: 4
```


In [ ]:
class Pokemon:
    def __init__(self, name, type_, hp, power):
        self.name = name
        self.type = type_
        self.hp = hp
        self.power = power

    def attack(self, target):
        multiplier = get_multiplier(self.type, target.type)   # 상성 배수
        damage = int(self.power * multiplier)                 # 데미지 계산
        target.hp -= damage                                   # 상대 HP 감소

        print(f"{self.name}의 공격! {target.name}에게 {damage} 데미지")
        if multiplier > 1.0:
            print("효과가 굉장했다")
        elif multiplier < 1.0:
            print("효과는 별로였다")
        else:
            print("그럭저럭 통했다")
        print(f"{target.name}의 남은 HP: {target.hp}")

# 확인
pikachu = Pokemon("피카츄", Type.ELECTRIC, 35, 20)
squirtle = Pokemon("꼬부기", Type.WATER, 44, 15)
pikachu.attack(squirtle)     # 전기 -> 물 = 2배


## 5. 개념 ③ random — 급소(크리티컬) 만들기

배틀에서 가끔 데미지가 더 크게 들어가는 '급소'를 넣어 보자. 이런 우연은 `random` 모듈로 만든다.

- `random.randint(1, 6)` → 1~6 중 정수 하나
- `random.random()` → 0.0 이상 1.0 미만의 실수
- `random.choice(목록)` → 목록에서 하나

확률을 만들 때는 `random.random()` 을 쓴다.
`random.random() < 0.1` 은 약 10% 확률로 참이 된다.
0~1 사이 숫자가 0.1보다 작을 확률이 10분의 1이기 때문이다.


In [ ]:
import random

print(random.randint(1, 6))      # 주사위
print(random.random())           # 0.0 ~ 1.0

# 10% 확률이 정말 가끔만 나오는지 확인
for i in range(10):
    if random.random() < 0.1:
        print(i, "급소")
    else:
        print(i, "보통")


## 6. 개념 ④ @dataclass — 클래스를 간단하게

4번에서 만든 클래스의 `__init__` 안에는 `self.name = name` 같은 줄이 반복된다.
**`@dataclass`** 를 클래스 위에 붙이면 파이썬이 `__init__` 을 자동으로 만들어 준다.
우리는 담을 정보만 적으면 된다.

```python
@dataclass
class Pokemon:
    name: str
    type: Type
    hp: int
    power: int
```

`name: str` 처럼 옆에 `: 종류` 를 적는 것을 **타입 힌트**라고 한다.
값의 종류를 알려 주는 메모이며, `@dataclass` 는 이 정보를 보고 `__init__` 을 만든다.

여기에 앞에서 배운 급소까지 넣어 포켓몬을 완성한다.


예상 출력 (급소가 터지면 데미지 1.5배, 40 → 60)
```
피카츄의 공격! 꼬부기에게 60 데미지
효과가 굉장했다
급소!
꼬부기의 남은 HP: -16
```


In [ ]:
from dataclasses import dataclass
import random

@dataclass                # __init__ 을 자동으로 만들어 준다
class Pokemon:
    name: str
    type: Type
    hp: int
    power: int

    def attack(self, target):
        multiplier = get_multiplier(self.type, target.type)
        damage = int(self.power * multiplier)

        is_critical = random.random() < 0.1   # 약 10% 확률로 급소
        if is_critical:
            damage = int(damage * 1.5)

        target.hp -= damage
        print(f"{self.name}의 공격! {target.name}에게 {damage} 데미지")
        if multiplier > 1.0:
            print("효과가 굉장했다")
        elif multiplier < 1.0:
            print("효과는 별로였다")
        else:
            print("그럭저럭 통했다")
        if is_critical:
            print("급소!")
        print(f"{target.name}의 남은 HP: {target.hp}")

# __init__ 을 적지 않아도 잘 만들어진다
pikachu = Pokemon("피카츄", Type.ELECTRIC, 35, 20)
squirtle = Pokemon("꼬부기", Type.WATER, 44, 15)
pikachu.attack(squirtle)


## 7. 직접 해보기 ② — 끝까지 싸우는 배틀

지금까지 만든 것을 모두 합치는 문제다.
두 포켓몬이 번갈아 공격해서, 한쪽 HP가 0 이하가 되면 끝나는 `battle(a, b)` 를 완성해 보자.

생각할 점
- 언제까지 반복할까 → 두 포켓몬의 HP가 모두 0보다 클 동안.
- a가 공격해서 b가 쓰러지면, b는 반격하지 못하고 끝나야 한다.
- 끝나면 HP가 남은 쪽이 승자다.

필요한 도구는 모두 배웠다: while 반복문, `attack`, if와 break.


In [ ]:
def battle(a, b):
    turn = 1
    while a.hp > 0 and b.hp > 0:     # 둘 다 살아있는 동안
        print(f"--- {turn}턴 ---")
        a.attack(b)
        if b.hp <= 0:                # b가 쓰러지면 반격 없이 종료
            break
        b.attack(a)
        turn += 1
    winner = a if a.hp > 0 else b    # HP 남은 쪽이 승자
    print(f"\n승자: {winner.name}")


## 8. 배틀 실행

피카츄와 꼬부기를 만들어 `battle` 로 싸워 본다.
급소가 들어 있어 실행할 때마다 결과가 조금씩 달라진다. 여러 번 실행해 보자.


예상 출력 (급소 때문에 매번 조금씩 다름)
```
--- 1턴 ---
피카츄의 공격! 꼬부기에게 40 데미지
효과가 굉장했다
꼬부기의 남은 HP: 4
꼬부기의 공격! 피카츄에게 15 데미지
그럭저럭 통했다
피카츄의 남은 HP: 20
--- 2턴 ---
피카츄의 공격! 꼬부기에게 40 데미지
효과가 굉장했다
꼬부기의 남은 HP: -36

승자: 피카츄
```


In [ ]:
pikachu = Pokemon("피카츄", Type.ELECTRIC, 35, 20)
squirtle = Pokemon("꼬부기", Type.WATER, 44, 15)
battle(pikachu, squirtle)


## 9. 마무리와 숙제

오늘 배운 것
- **Enum**: 정해진 값만 안전하게 쓰기
- **중첩 딕셔너리와 `.get()`**: 상성표 정리, 없는 조합은 기본값으로
- **클래스**: 포켓몬과 `attack`
- **random**: 급소 확률
- **@dataclass**: 클래스를 간단하게 만들기

숙제
1. 좋아하는 포켓몬 2~3마리를 더 만들고 `battle` 로 싸워 보기.
2. 상성표(`TYPE_CHART`)에 빠진 조합을 2개 이상 채우기. (예: 전기 → 풀은 몇 배일까)
3. (선택) 급소가 터지면 데미지를 1.5배 대신 2배로 바꿔 보고, 배틀이 어떻게 달라지는지 관찰하기.

다음 시간: 이 배틀을 파이게임 화면에 올려서 HP 바와 포켓몬 그림으로 보여 준다.
